# 🦜 LangChain 입문 실습
### AI 개발 강의 3주차 — LangChain 기초

---

## 📌 오늘 강의 목표

1. **지난주 복습**: OpenAI API를 직접 호출할 때의 불편함 인식
2. **LangChain이 왜 필요한가** 이해하기
3. LangChain 핵심 3요소 실습: **PromptTemplate / ChatModel / OutputParser**
4. **Chain 연결** — `invoke()` 단계별 방식으로 개념 이해
5. **LCEL** — 파이프(`|`) 문법으로 체인을 한 줄로 줄이기
6. **Streaming** — 응답을 실시간으로 출력하기



---
## 0️⃣ 환경 설정

In [ ]:
# 필요한 패키지 설치
!pip install langchain langchain-openai python-dotenv

In [ ]:
from dotenv import load_dotenv
# 같은 위치의 .env 파일을 메모리에 로드하는 함수
load_dotenv(override=True)


---
## 1️⃣ 지난주 방식 vs LangChain 방식 비교

### 🤔 왜 LangChain이 필요한가?

지난주에는 OpenAI API를 **직접** 호출했습니다.  
실제 서비스를 만들다 보면 어떤 문제가 생길까요?

| 상황 | 직접 API 호출 방식의 불편함 |
|------|-----------------------------|
| 프롬프트가 길어질 때 | 문자열 관리가 복잡해짐 |
| 모델을 바꾸고 싶을 때 | 코드 여러 곳을 수정해야 함 |
| 여러 단계를 연결할 때 | 직접 변수를 넘겨가며 연결해야 함 |
| 결과를 가공하고 싶을 때 | 매번 파싱 코드를 짜야 함 |

**LangChain**은 이런 반복 작업을 표준화된 방식으로 해결해주는 프레임워크입니다. 

(처음 진입시 문법이 어려운 이유 : 추상화 너무 많음)

#### OpenAI 방식

In [ ]:
# 지난주 방식 — OpenAI 직접 호출
from openai import OpenAI
client = OpenAI()

# 번역 기능을 만든다고 가정
language = "일본어"
text = "안녕하세요, 반갑습니다."

response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "user", 
            "content": f"{text}를 {language}로 번역해줘"
            }
    ]
)

result = response.choices[0].message.content
print(result)

#### Anthropic 방식 (참고)

In [ ]:
import anthropic
client = anthropic.Anthropic()

language = "일본어"
text = "안녕하세요, 반갑습니다."

# Claude 호출 (messages.create 사용)
response = client.messages.create(
    model="claude-3-5-sonnet-20240620", # 모델명 지정
    max_tokens=1024,                   # Anthropic은 이 값을 필수로 요구하는 경우가 많습니다
    messages=[
        {
            "role": "user", 
            "content": f"{text}를 {language}로 번역해줘"
        }
    ]
)

# 결과 추출 (OpenAI와 경로가 다름)
result = response.content[0].text
print(result)

#### Langchain 방식 (모델, 시스템 프롬프트등 변경시 소스를 많이 수정하지 않아도 되는 편리함)

In [ ]:
# LangChain 방식
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
#from langchain_anthropic import ChatAnthropic # 라이브러리 변경

prompt = PromptTemplate.from_template("{text}를 {language}로 번역해줘")
llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=5000)
#llm = ChatAnthropic(model="claude-3-5-sonnet-20240620") # LLM 모델 변경
parser = StrOutputParser()

# 세 요소를 연결 (invoke 방식)
formatted = prompt.invoke({"text": "안녕하세요, 반갑습니다.", "language": "일본어"})
response = llm.invoke(formatted)
result = parser.invoke(response)

print(result)

---
## 2️⃣ 핵심 요소 1: PromptTemplate

### 📌 개념

프롬프트를 **재사용 가능한 템플릿**으로 만드는 도구입니다.  
Python의 f-string과 비슷하지만, 변수 검증/재사용/관리가 훨씬 편합니다.

```
"{topic}에 대해 {style} 스타일로 설명해줘"
   ↑               ↑
 변수 자리        변수 자리
```

---

In [ ]:
from langchain_core.prompts import PromptTemplate

# 템플릿 정의  (template 선언) 
template = PromptTemplate.from_template(
    "{topic}에 대해 {style} 스타일로 3줄 요약해줘"
)

In [ ]:
# 변수 주입 1
filled1 = template.invoke({
    "topic": "인공지능",
    "style": "초등학생도 이해할 수 있는"
})

print("완성된 프롬프트:")
print(filled1.text)  # PromptTemplate 은 StringPromptValue 반환 → .text 사용

---

In [ ]:
# 같은 템플릿, 다른 변수 — 재사용성 확인
# 변수 주입 2
filled2 = template.invoke({
    "topic": "블록체인",
    "style": "개발자를 위한 기술적인"
})

print("완성된 프롬프트:")
print(filled2.text)

---

In [ ]:
# ChatPromptTemplate — 역할(role)을 나눠서 설정할 때
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다. 항상 한국어로 답변하세요."),
    ("human", "{question}")
])

filled3 = chat_template.invoke({
    "role": "Python 프로그래밍",
    "question": "리스트와 튜플의 차이가 뭔가요?"
})

print("📝 System:", filled3.messages[0].content)
print("💬 Human: ", filled3.messages[1].content)

### ✏️ 실습 1

아래 조건으로 `ChatPromptTemplate`을 직접 만들어보세요.

- system: `{domain}` 분야의 면접관 역할
- human: `{position}` 지원자에게 질문 1개 해줘

In [ ]:
##  실습 작성 칸
## 위에 보고 똑같이 작성해보셔도 됩니다.


* langchain_core.prompts  에서 MessagesPlaceholder 도 많이 사용함


---
## 3️⃣ 핵심 요소 2: ChatModel (LLM 래퍼)

### 📌 개념

모델마다 호출 방식이 전부 다릅니다.

```python
# OpenAI 직접 호출
response = client.chat.completions.create(...)
print(response.choices[0].message.content)  # 꺼내는 방법 A

# Anthropic 직접 호출
response = client.messages.create(...)
print(response.content[0].text)             # 꺼내는 방법 B

# Ollama 직접 호출
response = requests.post('http://localhost:11434/api/chat', ...)
print(response.json()['message']['content']) # 꺼내는 방법 C
```

**LangChain 래퍼**를 쓰면 모델이 바뀌어도 나머지 코드는 그대로입니다.

```
ChatOpenAI()    →  OpenAI GPT
ChatAnthropic() →  Claude
ChatOllama()    →  로컬 모델 (Ollama)
      ↑
선언부 한 줄만 바꾸면 — 나머지 코드 완전 동일
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# ── LangChain 래퍼로 모델 선언
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,  # 창의성 (0=일관, 1=창의)
    max_tokens=300
)

# 호출 방법은 항상 동일 — .invoke()
response = llm.invoke("파이썬을 배워야 하는 이유를 한 문장으로 말해줘")

print("📨 응답 타입:", type(response))
print("💬 내용:", response.content)
print()
print("👉 결과가 AIMessage 타입입니다. 문자열이 아니에요.")
print("   .content 로 텍스트를 꺼내거나, 다음에 배울 OutputParser를 쓰면 됩니다.")


In [ ]:
# ✅ LLM 래퍼의 핵심 — 선언부 한 줄만 바꾸면 나머지 코드는 완전히 동일

question = "파이썬을 배워야 하는 이유를 한 문장으로 말해줘"

# 모델 1: gpt-4o-mini (가볍고 빠름)
llm_mini = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 모델 2: gpt-4o (더 강력)
llm_large = ChatOpenAI(model="gpt-4o", temperature=0)
#llm_claude = ChatAnthropic(model="claude-3-5-sonnet-20240620")

# 나머지 코드는 완전히 동일 ↓
print("[gpt-4o-mini]")
print(llm_mini.invoke(question).content)
print()
print("[gpt-4o]")
print(llm_large.invoke(question).content)
print()
print("─" * 50)
print("💡 모델이 달라졌지만 .invoke() 코드는 한 글자도 안 바뀌었습니다.")
print()

# 실무에서는 이렇게도 교체 가능 (주석 — 별도 설치/API 키 필요)
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-3-haiku-20240307")  # Claude로 교체
#
# from langchain_ollama import ChatOllama
# llm = ChatOllama(model="llama3.1:70b", base_url="http://사내서버IP:11434")  # 사내 서버
#
# → 위 두 경우도 .invoke(question).content 로 동일하게 동작


---
## 4️⃣ 핵심 요소 3: OutputParser

### 📌 개념

LLM의 응답(문자열)을 **원하는 형태**로 변환합니다.

```
AIMessage(content="결과입니다")  →  StrOutputParser  →  "결과입니다"
AIMessage(content='{"name":"홍길동"}') →  JsonOutputParser →  {"name": "홍길동"}
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# llm 응답을 문자열로 변환
response = llm.invoke("사과를 영어로 번역해줘")

print("파싱 전:", type(response), response)
print()
print("파싱 후:", type(parser.invoke(response)), parser.invoke(response))

In [ ]:
# 🔧 JSON 형태로 받기
from langchain_core.output_parsers import JsonOutputParser

json_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
json_parser = JsonOutputParser()

response = json_llm.invoke(
    "다음 정보를 JSON으로 반환해줘 (name, age, job 키 포함): 홍길동, 30세, 개발자. "
    "마크다운 없이 순수 JSON만 반환해줘."
)

parsed = json_parser.invoke(response)
print("파싱 결과:", parsed)
print("타입:", type(parsed))
print("이름:", parsed["name"])

---
## 5️⃣ 세 요소 연결하기 — Chain

### 📌 개념

`PromptTemplate → ChatModel → OutputParser` 를 순서대로 연결합니다.

```
입력값 딕셔너리
     ↓
PromptTemplate  (변수 → 완성된 프롬프트)
     ↓
  ChatModel     (프롬프트 → AI 응답)
     ↓
 OutputParser   (AI 응답 → 원하는 형태)
     ↓
   최종 결과
```

**먼저 단계별 invoke()로 흐름을 이해하고, 이후 LCEL 파이프 문법으로 줄이는 과정을 배웁니다.**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# --- 각 요소 정의 ---
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 전문 번역가입니다. 번역문만 출력하세요."),
    ("human", "다음 문장을 {language}로 번역해줘: {text}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# ✅ 방법 1: invoke() 단계별 호출 — 흐름 이해용
def translate_step_by_step(text: str, language: str) -> str:
    step1 = prompt.invoke({"text": text, "language": language})  # 프롬프트 완성
    step2 = llm.invoke(step1)                                     # LLM 호출
    step3 = parser.invoke(step2)                                  # 파싱
    return step3

print("[단계별 방식]")
print(translate_step_by_step("오늘 날씨가 정말 좋네요!", "영어"))

---
## LCEL — 파이프(`|`) 문법으로 체인 만들기

### 📌 개념

LCEL(LangChain Expression Language)은 **LangChain 공식 권장 방식**입니다.  
리눅스 파이프(`|`)처럼 요소들을 연결해 체인을 한 줄로 선언합니다.

```python
# 단계별 방식
step1 = prompt.invoke(input)
step2 = llm.invoke(step1)
step3 = parser.invoke(step2)

# LCEL 방식 — 완전히 동일한 동작
chain = prompt | llm | parser
result = chain.invoke(input)
```


> ⚠️ **[현재 상태 주의]** LCEL 파이프(`|`) 문법은 LangChain v0.3까지의 공식 권장 방식이었으나,  
> **LangChain 1.0(2025년 9월 출시)** 이후 에이전트 중심 구조로 재편되면서 주력에서 밀려난 상태입니다.  
> 당장 제거되지는 않지만(v2.0 이전까지 동작 보장)
> 그럼에도 기존 코드 읽기 / 오픈소스 참고 시 여전히 자주 등장하므로 문법 자체는 알아두면 유용합니다.

In [ ]:
# ✅ 방법 2: LCEL 파이프 방식 — 현재 공식 권장
chain = prompt | llm | parser

# invoke()로 실행 — 입력은 동일하게 딕셔너리
result = chain.invoke({"text": "오늘 날씨가 정말 좋네요!", "language": "영어"})
print("[LCEL 방식]")
print(result)

# 여러 언어로 테스트
for lang in ["일본어", "스페인어", "프랑스어"]:
    print(f"{lang}: {chain.invoke({'text': '반갑습니다!', 'language': lang})}")

In [ ]:
# 체인을 변수로 관리하면 재사용이 쉬움
# 프롬프트만 바꿔서 완전히 다른 체인을 만들 수 있음

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 요약 전문가입니다. 핵심만 3줄로 요약하세요."),
    ("human", "{text}")
])

summarize_chain = summarize_prompt | llm | parser

text = """
LangChain은 대규모 언어 모델(LLM)을 활용한 애플리케이션 개발을 위한 프레임워크입니다.
프롬프트 관리, 모델 연동, 체인 구성, 에이전트 구축 등 다양한 기능을 제공합니다.
OpenAI, Anthropic, Ollama 등 다양한 모델을 동일한 인터페이스로 사용할 수 있습니다.
최근에는 복잡한 에이전트 워크플로우를 위한 LangGraph로 발전하고 있습니다.
"""

print(summarize_chain.invoke({"text": text}))

### ✏️ 실습 2

LCEL 파이프 방식으로 아래 기능을 구현해보세요.

**요구사항**: `{topic}` 키워드를 받아서 블로그 제목 3개를 제안하는 `title_chain` 만들기  
완성 후 `title_chain.invoke({"topic": "파이썬 입문"})` 으로 실행

In [ ]:
# 여기에 작성하세요
# title_prompt = ChatPromptTemplate.from_messages([...])
# title_chain = title_prompt | llm | parser



---
## 6️⃣ Streaming — 응답을 실시간으로 출력하기

### 📌 개념

`.invoke()`는 응답이 **완성될 때까지 기다렸다가** 한 번에 반환합니다.  
`.stream()`은 응답이 **생성되는 즉시 조각(chunk)씩** 출력합니다.

```
invoke():  [-------- 3초 대기 --------] → "긴 응답 전체"
stream():  "긴" → " 응" → "답" → " 전" → "체"  (즉시 출력)
```

> 💡 LCEL 체인은 `.stream()` 을 자동으로 지원합니다. ChatGPT처럼 타이핑되는 효과가 이것입니다.

In [ ]:
# ✅ stream() 사용법 — LCEL 체인에서 바로 사용 가능
story_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 창의적인 작가입니다."),
    ("human", "{topic}에 대한 짧은 이야기를 써줘 (5문장)")
])

story_chain = story_prompt | llm | parser

print("[스트리밍 출력 — 실시간으로 나타납니다]")
print("-" * 40)

# stream()은 청크를 하나씩 yield 함
for chunk in story_chain.stream({"topic": "로봇과 인간의 우정"}):
    print(chunk, end="", flush=True)  # end=""로 줄바꿈 없이 이어 출력

print("\n" + "-" * 40)

---
## 7️⃣ 실전 예제 — 간단한 Q&A 챗봇

대화 히스토리를 유지하는 챗봇을 LCEL 방식으로 만들어봅니다.

(back-end : python / front-end : vue, react 등...생략)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LCEL 방식으로 챗봇 체인 구성
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser() 

# LCEL 파이프로 체인 구성
chat_chain = chat_prompt | llm | parser

chat_history = []

def chat(question: str) -> str:
    
    inputs = {"history": chat_history, "question": question}
    

    print("🤖: ", end="", flush=True)
    answer = ""
    for chunk in chat_chain.stream(inputs):
        print(chunk, end="", flush=True)
        answer += chunk
    print()

    
    # 히스토리 업데이트
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    return answer


In [ ]:
# 이어서 대화해보기 (이부분이 front-end여야함)
# 1. 내 이름은 Charles야. 기억해줘.
# 2. 내 이름이 뭐라고?
# 
chat("내 이름이 뭐라고?")

---
## 8️⃣ 오늘 배운 것 정리

```
LangChain의 핵심 3요소 + LCEL

┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  PromptTemplate │ |  │    ChatModel    │ |  │  OutputParser   │
│                 │    │                 │    │                 │
│ 변수를 받아서     │    │ 프롬프트를 받아   │    │   AI 응답을 원하  │
│ 프롬프트 완성     │    │ AI에게 전달      │    │   는 형태로 변환  │
└─────────────────┘    └─────────────────┘    └─────────────────┘
        chain = prompt | llm | parser  ← LCEL (공식 권장)
```

| 개념 | 설명 | 주요 클래스/방법 |
|------|------|------------------|
| PromptTemplate | 재사용 가능한 프롬프트 틀 | `PromptTemplate`, `ChatPromptTemplate` |
| ChatModel | LLM 래퍼 (모델 교체 용이) | `ChatOpenAI`, `ChatOllama` |
| OutputParser | 응답 형태 변환 | `StrOutputParser`, `JsonOutputParser` |
| LCEL | 파이프로 체인 연결  | `prompt \| llm \| parser` |
| invoke() | 체인 실행 — 완성 후 반환 | `chain.invoke({...})` |
| stream() | 체인 실행 — 실시간 출력 | `chain.stream({...})` |
| MessagesPlaceholder | 대화 히스토리 주입 | `MessagesPlaceholder` |



---
## 9️⃣ 공지사항

1. 차주는 강의 없음 (PM교육 입과) 차차주에 만나요~ (수,금 투표)
2. 동영상은 올리는곳 찾아보는 중